In [1]:
from pyCHX.chx_packages import *

plt.rcParams.update({'figure.max_open_warning': 0})
plt.rcParams.update({ 'image.origin': 'lower'   })
plt.rcParams.update({ 'image.interpolation': 'none'   })

from time import strftime, localtime
from termcolor import colored

import sys
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/archiver_setup/")
from archiver_setup import *

%run /nsls2/data/chx/shared/CHX_Software/packages/environment_management/chx_analysis_setup.ipynb

/nsls2/conda/envs/2024-2.0-py311-tiled/lib/python3.11/site-packages/databroker/v1.py:72: UserWarning: In databroker 2.x, there are separate notions of 'server' and 'client', and register_handler(...) has no effect on the client. Likely this is being done for you on the server side, so you should not worry about this message unless you encounter trouble loading large array data.
  warnings.warn(


successfully imported CHX channel archiver and related functions
running on: jupyter_hub   environment: standard

setting '_base_path_' as /nsls2/data/chx/legacy/analysis/
setting '_base_path_pass_' as /nsls2/data/chx/proposals/
setting '_mask_path_' as /nsls2/data/chx/shared/CHX_Setup/Detector_masks/

ran "%run -i /nsls2/data/chx/shared/CHX_Software/packages/patches/roi_nr_2019_3_0_1.py" to fix ROI numbering for PHI-sliced data sets. Should get fixed in pyCHX...
ran "%run -i /nsls2/data/chx/shared/CHX_Software/packages/patches/chx_outlier_detection.py": this should become part of pyCHX.
ran "%run -i /nsls2/data/chx/shared/CHX_Software/packages/patches/fix_get_sid_filenames.py": this should be fixed and re-deployed in pyCHX.

environment dependent settings and patches:
using "%matplotlib inline" for plotting
ran "%run -i /nsls2/data/chx/shared/CHX_Software/packages/patches/polygonmask_fix.py".
ran "%run -i /nsls2/data/chx/shared/CHX_Software/packages/pyCHX/backups/chx_compress_05012024

### definition of beamline elements

In [2]:
class slit_gap_offset:
    def __init__(self, device, name):
        self.name=name
        self.xg = device+'-Ax:X}t2.C'
        self.xc = device+'-Ax:X}t2.D'
        self.yg = device+'-Ax:Y}t2.C'
        self.yc = device+'-Ax:Y}t2.D'
        self.attributes=['xg','xc','yg','yc']       

mbs = slit_gap_offset('XF:11IDA-OP{Slt:MB',name='mbs')
bds = slit_gap_offset('XF:11IDB-OP{Slt:BDS', name='bds')
gs =slit_gap_offset('XF:11IDB-OP{Slt:Guard', name='gs')

In [3]:
class XYMotor:
    def __init__(self, device, name):
        self.name=name
        self.x = device+ '-Ax:X}Mtr.RBV'
        self.y = device+ '-Ax:Y}Mtr.RBV'
        self.attributes=['x','y']

saxs_det = XYMotor('XF:11IDB-ES{Det:SAXS', name='saxs_det')

In [4]:
class hdm:
    def __init__(self, device, name):
        self.name=name
        self.x = device+'-Ax:X}Mtr.RBV'
        self.y = device+'-Ax:Y}Mtr.RBV'
        self.p = device+'-Ax:P}Pos-I'
        self.V = device+'-Ax:P}E-I'
        self.attributes=['x','y','p','V']
        
hdm = hdm('XF:11IDA-OP{Mir:HDM',name='hdm')

In [5]:
class Lslit:
    def __init__(self, device, name):
        self.name=name
        self.xg=device+'-Ax:XGap}Mtr.RBV'
        self.xc=device+'-Ax:XCtr}Mtr.RBV'
        self.yg=device+'-Ax:YGap}Mtr.RBV'
        self.yc=device+'-Ax:YCtr}Mtr.RBV'
        self.attributes=['xg','xc','yg','yc']
        
pbs = Lslit('XF:11IDA-OP{Slt:PB',name='pbs')
s1 = Lslit('XF:11IDB-OP{Slt:1',name='s1')

In [6]:
class double_crystal_monochromator:
    def __init__(self, device, name):
        self.name=name
        self.E=device+'-Ax:Energy}Mtr.RBV'
        self.b=device+'-Ax:B}Mtr.RBV'
        self.x=device+'-Ax:X}Mtr.RBV'
        self.r=device+'-Ax:R}Mtr.RBV'
        self.fp=device+'-Ax:FP}Mtr.RBV'
        self.attributes=['E','b','x','r','fp']
        
dcm = double_crystal_monochromator('XF:11IDA-OP{Mono:DCM',name='dcm')

In [7]:
class transfocator:
    def __init__(self, device, name):
        self.name=name
        self.l1=device+':1-Ax:X}Mtr.RBV'
        self.l2=device+':2-Ax:X}Mtr.RBV'
        self.l3=device+':3-Ax:X}Mtr.RBV'
        self.l4=device+':4-Ax:X}Mtr.RBV'
        self.l5=device+':5-Ax:X}Mtr.RBV'
        self.l6=device+':6-Ax:X}Mtr.RBV'
        self.l7=device+':7-Ax:X}Mtr.RBV'
        self.l8=device+':8-Ax:X}Mtr.RBV'
        self.x=device+':Ves-Ax:X}Mtr.RBV'
        self.y=device+':Ves-Ax:Y}Mtr.RBV'
        self.z=device+':Ves-Ax:Z}Mtr.RBV'
        self.ph=device+':Ves-Ax:Ph}Mtr.RBV'
        self.th=device+':Ves-Ax:Th}Mtr.RBV'        
        self.attributes=['l1','l2','l3','l4','l5','l6','l7','l8','x','y','z','ph','th']
            
trans=transfocator('XF:11IDA-OP{Lens',name='trans')

In [8]:
class diffractometer:
    def __init__(self, device, name):
        self.name = name
        self.Del= device+ '-Ax:Del}Mtr.RBV'
        self.gam = device+ '-Ax:Gam}Mtr.RBV'
        self.om = device+ '-Ax:Om}Mtr.RBV'
        self.phi = device+ '-Ax:Ph}Mtr.RBV'
        self.xb = device+ '-Ax:XB}Mtr.RBV'
        self.yb = device+ '-Ax:YB}Mtr.RBV'
        self.chh = device+ '-Ax:ChH}Mtr.RBV'
        self.thh = device+ '-Ax:ThH}Mtr.RBV'
        self.phh = device+ '-Ax:PhH}Mtr.RBV'
        self.xh = device+ '-Ax:XH}Mtr.RBV'
        self.yh = device+ '-Ax:YH2}Mtr.RBV'
        self.zh = device+ '-Ax:ZH}Mtr.RBV'
        self.chv = device+ '-Ax:ChV}Mtr.RBV'
        self.phv = device+ '-Ax:ThV}Mtr.RBV'
        self.xv = device+ '-Ax:XV}Mtr.RBV'
        self.yv = device+ '-Ax:YV}Mtr.RBV'
        self.zv = device+ '-Ax:ZV}Mtr.RBV'
        self.xv2 = device+ '-Ax:XV2}Mtr.RBV'
        self.attributes = ['Del','gam','om','phi','xb','yb','chh','thh','phh','xh','yh','zh','chv','phv','xv','yv','zv','xv2']

diff = diffractometer('XF:11IDB-ES{Dif',name='diff')

In [9]:
class Kinoform:
    def __init__(self, device, name):
        self.name=name  
        self.z = device+ '-Ax:ZB}Mtr.RBV'
        self.x = device+  '-Ax:XB}Mtr.RBV'
        self.y = device+  '-Ax:YB}Mtr.RBV'
        self.chi = device+  '-Ax:Ch}Mtr.RBV'
        self.theta = device+  '-Ax:Th}Mtr.RBV'
        self.phi = device+  '-Ax:Ph}Mtr.RBV'
        self.lx = device+  '-Ax:XT}Mtr.RBV'
        self.ly = device+  '-Ax:YT}Mtr.RBV'
        self.attributes=['z','x','y','chi','theta','phi','lx','ly']

kl1 = Kinoform('XF:11IDB-OP{Lens:1',name='kl1')
kl2 = Kinoform('XF:11IDB-OP{Lens:1',name='kl2')

In [10]:
class SAXSBeamStop:
    def __init__(self, device, name):
        self.name=name 
        self.x = device+ '-Ax:X}Mtr.RBV' 
        self.y1 = device+ '-Ax:YFT}Mtr.RBV'
        self.x2 = device+ '-Ax:XFB}Mtr.RBV'
        self.y2 = device+ '-Ax:YFB}Mtr.RBV'
        self.attributes=['x','y1','x2','y2']

saxs_bst = SAXSBeamStop('XF:11IDB-ES{BS:SAXS',name='saxs_bst')

In [11]:
class SAXS_Table:
     def __init__(self, device, name):
        self.name=name 
        self.z = device+ '-Ax:Z}Mtr.RBV'
        self.gap = device+ '-Ax:Gap}Mtr.RBV'
        self.theta = device+ '-Ax:Theta}Mtr.RBV'
        self.waxs_skew = device+ '-Ax:Skew}Mtr.RBV'
        self.saxs_offset = device+ '-Ax:Offset}Mtr.RBV'
        self.saxs_skew = device+ '-Ax:Skew}Mtr.RBV'
        self.x1 = device+ '-Ax:X1}Mtr.RBV'
        self.x2 = device+ '-Ax:X2}Mtr.RBV'
        self.x3 = device+ '-Ax:X3}Mtr.RBV'
        self.x4 = device+ '-Ax:X4}Mtr.RBV'
        self.z1 = device+ '-Ax:Z1}Mtr.RBV'
        self.z2 = device+ '-Ax:Z2}Mtr.RBV'
        self.attributes = ['z','gap','theta','waxs_skew','saxs_offset','saxs_skew','x1','x2','x3','x4','z1','z2']
         
saxs_table = SAXS_Table('XF:11IDB-ES{Tbl:SAXS',name = 'saxs_table')

### setup of the CHX beamline

In [12]:
chx_beamline={'ivu':0,'wbs':1,'hdm':2,'pbs':3,'dcm':4,'mbs':6,'trans':7,'s1':8,'kl1':9,'kl2':10,'bds':11,'gs':12,'diff':13,
              'saxs_table':14,'saxs_bst':15,'saxs_det':16}

### select components for which you want to retrieve positions

In [16]:
comp_list=['pbs','hdm','mbs','dcm','trans','s1','kl1','kl2','bds','gs','diff','saxs_bst','saxs_det']

### select (maximum of 2) uids for comparison

In [17]:
uid_list=['8bff73fa-5390-4906-8c14-3b830c4187ce','b6899cbd-37d3-4dd2-9103-6ca6d51e6cc6']
uid_list =['f6f50fb9-4de6-4f86-b34f-5a15a71e8345']

#### retrieve data from archiver

In [18]:
bl_dict={}
for c in comp_list:
    bl_dict[chx_beamline[c]]={'name':c}
for c in bl_dict:
    attr_list = eval(bl_dict[c]['name']).attributes
    bl_dict[c]['attr']={}
    for a in attr_list:
        bl_dict[c]['attr'][a]={}
        bl_dict[c]['attr'][a]['pv']=eval(bl_dict[c]['name']+'.'+a)

## get MEAN values for uid
for c in list(bl_dict.keys()):
    for u in uid_list:
        for a in list(bl_dict[c]['attr'].keys()):
            #print(bl_dict[c]['attr'][a]['pv'])
            dd=get_archived_pvs_from_uid(pv_list=[bl_dict[c]['attr'][a]['pv']],uid=u,pre=10,post=10,verbose=False)
            bl_dict[c]['attr'][a][u]={}
            bl_dict[c]['attr'][a][u]['val']=np.nanmean(dd[bl_dict[c]['attr'][a]['pv']]['data'])

### print beamline positions

In [19]:
assert len(uid_list)<=2,'ERROR: can handle maximum of 2 uids at this time'
date_str=''
for u in uid_list:
    date_str+='\t'+u[:8];date_str+='\t\t\t'
date_str+='\n'
for u in uid_list:
    h=db[u]
    t=np.mean([h.start['time'],h.stop['time']])
    date = strftime('%Y-%m-%d %H:%M', localtime(t))
    date_str+=date;date_str+='\t\t\t'
if len(uid_list)==2:
    date_str+='DIFFERENCE';date_str+='\n'
print(colored(date_str,'green'))

for c in sorted(list(bl_dict.keys())):
    for a in list(bl_dict[c]['attr'].keys()):
        str_='';tmp=[]
        for u in uid_list:
            tmp.append(bl_dict[c]['attr'][a][u]['val'])
            str_+=bl_dict[c]['name']+'.'+a+' = %s\t\t\t'%np.round(bl_dict[c]['attr'][a][u]['val'],5)
        if len(uid_list)==2:
            str_+=str(np.round(tmp[0]-tmp[1],5))
        print(str_)
    print('\n')

	f6f50fb9			
2024-10-27 21:50			
hdm.x = 0.2492			
hdm.y = -2.4996			
hdm.p = -31.8832			
hdm.V = 39.22286			


pbs.xg = 0.60005			
pbs.xc = 0.0			
pbs.yg = 1.2			
pbs.yc = 0.0395			


dcm.E = 9648.0			
dcm.b = -11.83158			
dcm.x = 1.525			
dcm.r = -0.0265			
dcm.fp = 220.26779			


mbs.xg = 0.05002			
mbs.xc = -0.0124			
mbs.yg = 0.39991			
mbs.yc = 0.07495			


trans.l1 = 14.0674			
trans.l2 = -1e-05			
trans.l3 = 14.98849			
trans.l4 = 14.24668			
trans.l5 = -1e-05			
trans.l6 = 14.70437			
trans.l7 = 16.07696			
trans.l8 = 15.00336			
trans.x = -0.6506			
trans.y = -4.9028			
trans.z = -0.0062			
trans.ph = -0.73998			
trans.th = -0.4148			


s1.xg = 0.08001			
s1.xc = 1.58875			
s1.yg = 0.19999			
s1.yc = 0.1673			


kl1.z = -0.09945			
kl1.x = 0.57031			
kl1.y = 2.95041			
kl1.chi = 0.39977			
kl1.theta = -0.0399			
kl1.phi = -0.52283			
kl1.lx = -5.21822			
kl1.ly = -6.09957			


kl2.z = -0.09945			
kl2.x = 0.57031			
kl2.y = 2.95041			
kl2.chi = 0.39977			
kl2.theta = -0.0399		